In [1]:
import sys
sys.path.append("../src")
from utils import *

In [3]:
class Logic:
    def __init__(self, nin_data = 8, nin_control = 3):
        self.nin_data = nin_data
        self.nin_control = nin_control


        self.and_gates_of_input = [AND(2) for _ in range(self.nin_data)]
        self.xor_gates_of_input = [XOR() for _ in range(self.nin_data)]
        self.or_gates_of_input = [OR(2) for _ in range(self.nin_data)]

        self.and_gate_of_enable_and = AND(3)
        self.and_gate_of_enable_xor = AND(3)
        self.and_gate_of_enable_or = AND(3)

        self.invert_gate_of_f0 = Inverter()
        self.invert_gate_of_f1 = Inverter()

        self.enable_and_output = 0
        self.enable_xor_output = 0
        self.enable_or_output = 0

    def __call__(self, input_A, input_B, F2_0):
        assert len(input_A) == self.nin_data, "Input must be {self.nin_data} bits long"
        assert len(input_B) == self.nin_data, "Input must be {self.nin_data} bits long"
        assert len(F2_0) == self.nin_control, "Input must be {self.nin_control} bits long"

        invert_of_f0 = self.invert_gate_of_f0([F2_0[-1]])
        invert_of_f1 = self.invert_gate_of_f1([F2_0[-2]])

        self.enable_and_output = self.and_gate_of_enable_and([invert_of_f0, invert_of_f1, F2_0[0]])
        self.enable_xor_output = self.and_gate_of_enable_xor([F2_0[-1], invert_of_f1, F2_0[0]])
        self.enable_or_output = self.and_gate_of_enable_or([invert_of_f0, F2_0[1], F2_0[0]])

        output = []
        if self.enable_and_output:
            for idx, and_gate in enumerate(self.and_gates_of_input):
                output.append(and_gate([input_A[idx], input_B[idx]]))

        elif self.enable_xor_output:
            for idx, xor_gate in enumerate(self.xor_gates_of_input):
                output.append(xor_gate([input_A[idx], input_B[idx]]))
        elif self.enable_or_output:
            for idx, or_gate in enumerate(self.or_gates_of_input):
                output.append(or_gate([input_A[idx], input_B[idx]]))
        else:
            # If not selected (e.g., ALU is doing Addition), output 8 dead wires
            output = [0] * self.nin_data

        return output

In [ ]:
class Add_Subtract:
    def __init__(self, nin_data = 8, nin_control = 2):
        self.nin_data = nin_data
        self.nin_control = nin_control

        # ones' complement
        self.xor_gates_for_complements = [XOR() for _ in range(self.nin_data)]

        # NBitsAccumulator
        self.adder = NBitAdderWithOverflow(self.nin_data)

        self.invert_gate_of_f0 = Inverter()
        self.and_gate_of_f0_CY_in = AND(2)
        self.and_gate_of_f0_f1 = AND(2)

        self.or_gate_of_CI = OR(2)

    def __call__(self, input_A, input_B, F1_0, CY_In = 0):
        assert len(input_A) == self.nin_data, f"Input must be {self.nin_data} bits long"
        assert len(input_B) == self.nin_data, f"Input must be {self.nin_data} bits long"
        assert len(F1_0) == self.nin_control, f"Input must be {self.nin_control} bits long"

        invert_of_f0 = self.invert_gate_of_f0([F1_0[1]])
        invert_flag = F1_0[0]
        CI_flag = self.or_gate_of_CI([self.and_gate_of_f0_CY_in([F1_0[1], CY_In]), self.and_gate_of_f0_f1([invert_of_f0, F1_0[0]])])
        
        outputs_from_complements_of_one = []
        for idx, xor_gate in enumerate(self.xor_gates_for_complements):
            outputs_from_complements_of_one.append(xor_gate([invert_flag, input_B[idx]]))
        operand_B = outputs_from_complements_of_one

        overflow, sum_bits = self.adder(input_A, operand_B, CI_flag)
        carry_out = self.adder.get_carry_out()

        return carry_out, sum_bits
        
            
    

In [ ]:
class ALU:
    def __init__(self, nin_data = 8, nin_control_signal = 3):
        self.nin_data = nin_data
        self.nin_control = nin_control_signal

        self.add_and_subtract = Add_Subtract(self.nin_data, self.nin_control-1)
        self.logic = Logic(self.nin_data, self.nin_control)

        self.enable_add_or_sub = 0

        # just use 3 bits although use nin_data bits here
        self.flags_latch = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nin_data)
        self.result_latch = NBitsEdgeTriggeredDTypeFlipFlopWithPresetAndClear(self.nin_data)

        self.invert_of_f2 = Inverter()
        self.and_gate_of_f0_f1 = AND()
        self.or_gate_of_invert_of_f2_other = OR()
        self.and_gate_of_cy_out_other = AND()

        self.nor_gates_of_flags =  NOR(self.nin_data)

    def __call__(self, input_A, input_B, F2_0, clock, enable):
        assert len(input_A) == self.nin_data, f"Input must be {self.nin_data} bits long"
        assert len(input_B) == self.nin_data, f"Input must be {self.nin_data} bits long"
        assert len(F2_0) == self.nin_control, f"Input must be {self.nin_control} bits long"
        assert clock in [0, 1], f"clock signal must be 0 or 1"
        assert enable in [0, 1], f"enable signal must be 0 or 1"

        invert_of_f2 = self.invert_of_f2([F2_0[0]])
        self.enable_add_or_sub = invert_of_f2

        and_of_f1_f0 = self.and_gate_of_f0_f1([F2_0[1], F2_0[2]])
        or_of_and_invert_f2 = self.or_gate_of_invert_of_f2_other([invert_of_f2, and_of_f1_f0])

        # flags current have 3 bits is meaningful: 
        # flags[0]: sign flag, flags[1]: zero flag, flags[2]: carry flag
        flags = self.flags_latch.getQ()
        # Carry flag then circles back up to provide the CY In input of the Add/Sub module
        carry_out, sum_bits = self.add_and_subtract(input_A, input_B, F2_0[1:], flags[2])

        and_of_cy_out_or = self.and_gate_of_cy_out_other([carry_out, or_of_and_invert_f2])

        # new flags
        new_flags = [0] * self.nin_data
        new_flags[2] = and_of_cy_out_or

        result_of_logic = self.logic(input_A, input_B, F2_0)

        # this tick's output from add_and_subtract
        if self.enable_add_or_sub:
            new_flags[1] = self.nor_gates_of_flags(sum_bits)
            new_flags[0] = sum_bits[0] # MSB

            self.result_latch(sum_bits, clock)

        else:
            new_flags[1] = self.nor_gates_of_flags(result_of_logic)
            # new_flags[0] = flags[0] # old sign flag
            new_flags[0] = result_of_logic # maybe we also check logic output?

            self.result_latch(result_of_logic, clock)

        self.flags_latch(new_flags, clock)

        output = [0] * self.nin_data
        if enable:
            output = self.result_latch.getQ()

        return output


        




        